# Thí nghiệm v2: dữ liệu giàu hơn cho Naive Bayes

Notebook này **không đọc `datasets/v2/test_v2.csv`** và không đọc nội dung test cũ. Thiết kế khóa trước trong `experiment_plan_v2.md`.

Pool dữ liệu ngoài (UIT-VSMEC, UIT-VSFC) chỉ nằm ở phần train của fold. **Điểm CV chỉ tính trên dòng phim, với nhãn gốc.**

Các ô đầu dùng lại nguyên harness đã kiểm chứng của `naive_bayes_improvement.ipynb`, nên fold và các phép kiểm tra đúng đắn giữ nguyên.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import hashlib, itertools, json, math, platform, re, subprocess, sys, time
from collections import Counter, namedtuple
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sps
import sklearn
import torch
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda")
DTYPE = torch.float64
N_JOBS = len(os.sched_getaffinity(0))

ROOT = Path.cwd()
while not (ROOT / "Data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SPLITS = ROOT / "Data" / "splits"
XLSX = ROOT / "Data" / "final_labeled_clean_v3.xlsx"
NBDIR = ROOT / "models" / "Naive_Bayes"
EXP = NBDIR / "experiments"
(EXP / "oof").mkdir(parents=True, exist_ok=True)

LABELS = ["Negative", "Neutral", "Positive"]
L2I = {l: i for i, l in enumerate(LABELS)}
C = len(LABELS)

LOCKED = {
    "Data/splits/train.csv": "c04400673765183a0c904d447b611f48b74389c1b68059d517237d4cdab38505",
    "Data/splits/val.csv": "155c13a615172a7b06c4f2b647e222736d0354dcb603ad90c0c027ecd67f31c7",
    "Data/splits/test.csv": "6135fb08b390b6743968a1fb1dde8319977e3297bad656f00e1d669f972b9ae2",
    "Data/final_labeled_clean_v3.xlsx": "ea6c578763a8396f4c065696e50df0431a0a978cfff40399818e2a7b4ef7a89c",
}
for rel, want in LOCKED.items():
    got = hashlib.sha256((ROOT / rel).read_bytes()).hexdigest()
    assert got == want, f"File bị thay đổi: {rel}"

print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "| sklearn", sklearn.__version__,
      "| numpy", np.__version__, "| workers", N_JOBS, "| OMP", os.environ["OMP_NUM_THREADS"])
print("Hash khớp cho", len(LOCKED), "file bị khóa")

torch 2.8.0+cu128 | NVIDIA A100-PCIE-40GB | sklearn 1.6.1 | numpy 2.0.2 | workers 8 | OMP 1
Hash khớp cho 4 file bị khóa


In [2]:
EXP = NBDIR / "experiments_v2"
(EXP / "oof").mkdir(parents=True, exist_ok=True)
print("Kết quả ghi vào", EXP.relative_to(ROOT))

Kết quả ghi vào models/Naive_Bayes/experiments_v2


In [3]:
def read_split(name):
    return pd.read_csv(SPLITS / f"{name}.csv", encoding="utf-8-sig", keep_default_na=False)

dev = pd.concat([read_split("train").assign(origin="train"), read_split("val").assign(origin="val")], ignore_index=True)
xl = pd.read_excel(XLSX, keep_default_na=False, usecols=["Comment ID", "RAW Text", "Clean Text", "Final Label", "Clean Text Old"])
xl = xl[xl["Comment ID"].isin(dev["Comment ID"])].set_index("Comment ID").loc[dev["Comment ID"]].reset_index()
assert (xl["Clean Text"].astype(str).values == dev["Clean Text"].astype(str).values).all()
assert (xl["Final Label"].values == dev["Final Label"].values).all()
dev["Clean Text Old"] = xl["Clean Text Old"].astype(str).values
dev["RAW Text"] = xl["RAW Text"].astype(str).values
del xl

N = len(dev)
y_np = dev["Final Label"].map(L2I).to_numpy()
gid_np = pd.factorize(dev["Clean Text"])[0]
y = torch.tensor(y_np, device=DEVICE)
gid = torch.tensor(gid_np, device=DEVICE)
is_train = torch.tensor((dev["origin"] == "train").to_numpy(), device=DEVICE)
print("dev:", N, "| group:", int(gid.max()) + 1, "|", dev["Final Label"].value_counts().to_dict())

R, K, Q = 5, 5, 3
F_np = np.zeros((R, N), dtype=np.int64)
I_np = np.zeros((R, K, N), dtype=np.int64)
dummy = np.zeros(N)
for r in range(R):
    sgkf = StratifiedGroupKFold(K, shuffle=True, random_state=r)
    for k, (_, te) in enumerate(sgkf.split(dummy, y_np, gid_np)):
        F_np[r, te] = k
    for k in range(K):
        idx = np.where(F_np[r] != k)[0]
        inner = StratifiedGroupKFold(Q, shuffle=True, random_state=100 * r + k)
        for q, (_, te) in enumerate(inner.split(dummy[idx], y_np[idx], gid_np[idx])):
            I_np[r, k, idx[te]] = q

for r in range(R):
    for k in range(K):
        a, b = set(gid_np[F_np[r] == k]), set(gid_np[F_np[r] != k])
        assert not (a & b), "group lẫn qua outer fold"
        idx = np.where(F_np[r] != k)[0]
        for q in range(Q):
            a, b = set(gid_np[idx[I_np[r, k, idx] == q]]), set(gid_np[idx[I_np[r, k, idx] != q]])
            assert not (a & b), "group lẫn qua inner fold"
print("Không có group nào lẫn qua outer/inner fold")

Ft, It = torch.tensor(F_np, device=DEVICE), torch.tensor(I_np, device=DEVICE)
pd.DataFrame({"Comment ID": dev["Comment ID"], **{f"fold_r{r}": F_np[r] for r in range(R)}}).to_csv(EXP / "cv_folds.csv", index=False)

dev: 1800 | group: 1748 | {'Positive': 1091, 'Neutral': 392, 'Negative': 317}


Không có group nào lẫn qua outer/inner fold


In [4]:
TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")          # cùng quy ước với sklearn và notebook gốc

def tokens(text):
    return TOKEN_RE.findall(text.lower())

def word_terms(toks):
    return [(w, 1) for w in toks] + [(f"{toks[i]} {toks[i + 1]}", 2) for i in range(len(toks) - 1)]

def char_wb_terms(text, lo=2, hi=5):
    # cùng thuật toán với analyzer='char_wb' của sklearn; '_' (dấu tách từ ghép) đổi thành khoảng trắng
    out = []
    for w in text.lower().replace("_", " ").split():
        w = f" {w} "
        for n in range(lo, hi + 1):
            off = 0
            out.append((w[off:off + n], n))
            while off + n < len(w):
                off += 1
                out.append((w[off:off + n], n))
            if off == 0:
                break
    return out

class Corpus:
    def __init__(self, term_lists):
        idx, order, d_, t_, f_ = {}, [], [], [], []
        for i, terms in enumerate(term_lists):
            for (w, o), c in Counter(terms).items():
                j = idx.get(w)
                if j is None:
                    j = idx[w] = len(idx)
                    order.append(o)
                d_.append(i); t_.append(j); f_.append(c)
        self.vocab = list(idx)
        self.doc = torch.tensor(d_, device=DEVICE)
        self.term = torch.tensor(t_, device=DEVICE)
        self.tf = torch.tensor(f_, dtype=DTYPE, device=DEVICE)
        self.order = torch.tensor(order, device=DEVICE)
        self.n_docs, self.n_terms = len(term_lists), len(idx)

Feat = namedtuple("Feat", "d t v vm")

def vectorize(cp, tr, min_df=2, max_order=2, weighting="tfidf", keep=None, df_mask=None):
    dfm = tr if df_mask is None else df_mask
    ok = cp.order[cp.term] <= max_order
    df = torch.bincount(cp.term[dfm[cp.doc] & ok], minlength=cp.n_terms).to(DTYPE)
    vm = df >= min_df
    if keep is not None:
        vm = vm & keep
    m = vm[cp.term]
    d, t, f = cp.doc[m], cp.term[m], cp.tf[m]
    if weighting == "tfidf":
        n = dfm.sum().to(DTYPE)
        idf = torch.log((1 + n) / (1 + df)) + 1.0
        v = (1.0 + torch.log(f)) * idf[t]
        norm2 = torch.zeros(cp.n_docs, dtype=DTYPE, device=DEVICE).index_add_(0, d, v * v)
        v = v / norm2.sqrt()[d].clamp_min(1e-12)
    elif weighting == "count":
        v = f
    elif weighting == "binary":
        v = torch.ones_like(f)
    else:
        raise ValueError(weighting)
    return Feat(d, t, v, vm)

def nb_weights(ft, yy, tr, alphas, kind="mnb", w=None):
    k = tr[ft.d]
    vals = ft.v[k] if w is None else ft.v[k] * w[ft.d[k]]
    fc = torch.zeros(C, ft.vm.numel(), dtype=DTYPE, device=DEVICE)
    fc.index_put_((yy[ft.d[k]], ft.t[k]), vals, accumulate=True)
    if kind == "cnb":
        fc = fc.sum(0, keepdim=True) - fc
    s = (fc[None] + alphas.view(-1, 1, 1)) * ft.vm
    lt = torch.log(s.clamp_min(1e-300)) - torch.log(s.sum(2, keepdim=True))
    return -lt if kind == "cnb" else lt

def nb_scores(ft, lt, ev):
    n_ev = int(ev.sum())
    pos = torch.full((ev.numel(),), -1, device=DEVICE)
    pos[ev] = torch.arange(n_ev, device=DEVICE)
    e = ev[ft.d]
    jl = torch.zeros(lt.shape[0], n_ev, C, dtype=DTYPE, device=DEVICE)
    jl.index_add_(1, pos[ft.d[e]], ft.v[e].view(1, -1, 1) * lt[:, :, ft.t[e]].transpose(1, 2))
    return jl

def log_prior(yy, tr, w=None):
    cc = torch.zeros(C, dtype=DTYPE, device=DEVICE)
    cc.index_add_(0, yy[tr], torch.ones(int(tr.sum()), dtype=DTYPE, device=DEVICE) if w is None else w[tr])
    return torch.log(cc / cc.sum())

def select_terms(ft, yy, tr, topk, method):
    # bảng 3×2 (lớp × có/không có term), tính chỉ trên văn bản train
    k = tr[ft.d]
    O = torch.zeros(C, ft.vm.numel(), dtype=DTYPE, device=DEVICE)
    O.index_put_((yy[ft.d[k]], ft.t[k]), torch.ones(int(k.sum()), dtype=DTYPE, device=DEVICE), accumulate=True)
    Nc = torch.bincount(yy[tr], minlength=C).to(DTYPE).view(C, 1)
    n = Nc.sum()
    A, B = O, Nc - O
    n1 = A.sum(0, keepdim=True)
    E1, E0 = Nc * n1 / n, Nc * (n - n1) / n
    if method == "chi2":
        s = ((A - E1) ** 2 / E1.clamp_min(1e-12) + (B - E0) ** 2 / E0.clamp_min(1e-12)).sum(0)
    else:
        xl = lambda a, e: torch.where(a > 0, (a / n) * torch.log(a / e.clamp_min(1e-12)), torch.zeros_like(a))
        s = (xl(A, E1) + xl(B, E0)).sum(0)
    s = s.masked_fill(~ft.vm, -1.0)
    keep = torch.zeros_like(ft.vm)
    keep[s.topk(min(topk, int(ft.vm.sum()))).indices] = True
    return keep

def macro_f1_cm(cm):
    tp = cm.diagonal(dim1=-2, dim2=-1)
    prec = tp / cm.sum(-2).clamp_min(1e-12)
    rec = tp / cm.sum(-1).clamp_min(1e-12)
    return 2 * prec * rec / (prec + rec).clamp_min(1e-12)

def cm_batch(yy, pred):                            # pred: (K, n) -> (K, C, C); hàng = nhãn thật
    code = yy.view(1, -1) * C + pred
    return (torch.zeros(pred.shape[0], C * C, dtype=DTYPE, device=DEVICE)
            .scatter_add_(1, code, torch.ones_like(code, dtype=DTYPE)).view(-1, C, C))

def score_macro(cm):
    return macro_f1_cm(cm).mean(-1)

CORPORA = {}
def corpus(key, build):
    if key not in CORPORA:
        CORPORA[key] = Corpus(build())
    return CORPORA[key]

T_CLEAN = [tokens(t) for t in dev["Clean Text"]]
corpus("clean", lambda: [word_terms(t) for t in T_CLEAN])
print("clean: n_terms", CORPORA["clean"].n_terms, "| nnz", CORPORA["clean"].doc.numel())

clean: n_terms 25891 | nnz 59501


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.naive_bayes import ComplementNB, MultinomialNB

cp = CORPORA["clean"]
tr_m, va_m = is_train, ~is_train
tr_txt = dev.loc[dev.origin == "train", "Clean Text"].tolist()
va_txt = dev.loc[dev.origin == "val", "Clean Text"].tolist()

ft = vectorize(cp, tr_m, 2, 2, "tfidf")
sk = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(tr_txt)
assert int(ft.vm.sum()) == len(sk.get_feature_names_out()) == 4204
vocab_np = np.array(cp.vocab)
colmap = {w: i for i, w in enumerate(sk.get_feature_names_out())}

def to_dense(ft_, mask):
    idx = torch.full((mask.numel(),), -1, device=DEVICE); idx[mask] = torch.arange(int(mask.sum()), device=DEVICE)
    e = mask[ft_.d]
    cols = np.array([colmap[w] for w in vocab_np[ft_.t[e].cpu().numpy()]])
    out = np.zeros((int(mask.sum()), len(colmap)))
    out[idx[ft_.d[e]].cpu().numpy(), cols] = ft_.v[e].cpu().numpy()
    return out

Xtr, Xva = to_dense(ft, tr_m), to_dense(ft, va_m)
print("TF-IDF max|diff| train:", np.abs(Xtr - sk.transform(tr_txt).toarray()).max(), "| val:", np.abs(Xva - sk.transform(va_txt).toarray()).max())

alphas = torch.tensor([0.01], dtype=DTYPE, device=DEVICE)
jl = nb_scores(ft, nb_weights(ft, y, tr_m, alphas), va_m)[0]
sk_nb = MultinomialNB(alpha=0.01, fit_prior=False).fit(Xtr, y_np[is_train.cpu().numpy()])
p = jl.argmax(1).cpu().numpy()
agree = (p == sk_nb.predict(Xva)).mean()
f1 = float(score_macro(cm_batch(y[va_m], jl.argmax(1)[None]))[0])
print("MNB trùng sklearn:", agree, "| macro-F1 val:", f1, "| sklearn:", f1_score(y_np[~is_train.cpu().numpy()], p, average="macro"))
assert agree == 1.0 and abs(f1 - 0.6146723400992851) < 1e-12, "không tái lập được kết quả val đã công bố"

jc = nb_scores(ft, nb_weights(ft, y, tr_m, alphas, "cnb"), va_m)[0]
sk_cnb = ComplementNB(alpha=0.01).fit(Xtr, y_np[is_train.cpu().numpy()])
print("CNB trùng sklearn:", (jc.argmax(1).cpu().numpy() == sk_cnb.predict(Xva)).mean(),
      "| max|diff| jll:", np.abs(jc.cpu().numpy() - sk_cnb.predict_joint_log_proba(Xva) ).max())

# bootstrap: trọng số theo group, đối chiếu naive
g = torch.Generator(device=DEVICE).manual_seed(SEED)
B_BOOT, G = 5000, int(gid.max()) + 1
draw = torch.randint(0, G, (B_BOOT, G), generator=g, device=DEVICE)
W_G = torch.zeros(B_BOOT, G, dtype=DTYPE, device=DEVICE).scatter_add_(1, draw, torch.ones_like(draw, dtype=DTYPE))
W_BOOT = W_G[:, gid]
del draw, W_G

def boot_scores(pred, yy=y, W=None):               # pred: (R, N) -> (B,)
    W = W_BOOT if W is None else W
    out = 0.0
    for r in range(pred.shape[0]):
        oh = torch.nn.functional.one_hot(yy * C + pred[r], C * C).to(DTYPE)
        out = out + score_macro((W @ oh).view(-1, C, C))
    return out / pred.shape[0]

rng = np.random.default_rng(0)
fake = torch.tensor(rng.integers(0, C, size=(1, N)), device=DEVICE)
fast = boot_scores(fake)
for b in [0, 1, 2]:
    w = W_BOOT[b].cpu().numpy().astype(int)
    naive = f1_score(np.repeat(y_np, w), np.repeat(fake[0].cpu().numpy(), w), average="macro")
    assert abs(naive - float(fast[b])) < 1e-12
for b in range(50):
    w = W_BOOT[b]
    assert all(w[gid == q].unique().numel() == 1 for q in gid.unique()[:30])
print("Bootstrap theo group khớp bản naive và trọng số đồng nhất trong group")

# chọn đặc trưng: chi2 trên bảng 3×2 khớp scipy
from scipy.stats import chi2_contingency
ft0 = vectorize(cp, tr_m, 2, 2, "tfidf")
keep = select_terms(ft0, y, tr_m, 50, "chi2")
some = keep.nonzero().ravel()[0].item()
docs = ft0.d[(ft0.t == some) & tr_m[ft0.d]]
A = torch.bincount(y[docs], minlength=C).cpu().numpy()
Nc = np.bincount(y_np[is_train.cpu().numpy()], minlength=C)
print("chi2 scipy cho một term:", round(chi2_contingency(np.vstack([A, Nc - A]), correction=False)[0], 6))
print("Tất cả kiểm tra đúng đắn đã qua")

TF-IDF max|diff| train: 2.220446049250313e-16 | val: 1.6653345369377348e-16
MNB trùng sklearn: 1.0 | macro-F1 val: 0.6146723400992851 | sklearn: 0.6146723400992852
CNB trùng sklearn: 1.0 | max|diff| jll: 9.947598300641403e-14


Bootstrap theo group khớp bản naive và trọng số đồng nhất trong group
chi2 scipy cho một term: 19.327931
Tất cả kiểm tra đúng đắn đã qua


In [6]:
ALPHAS = (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0)

class NBVariant:
    def __init__(self, name, corpora=("clean",), kind="mnb", weightings=("tfidf",), max_orders=(1, 2), min_dfs=(1, 2, 3),
                 alphas=ALPHAS, priors=(False, True), offsets=((0.0, 0.0),), topks=(None,), weight_fn=None, ext_src=None):
        self.name, self.kind, self.weight_fn = name, kind, weight_fn
        self.alphas = alphas
        self.alpha_t = torch.tensor(alphas, dtype=DTYPE, device=DEVICE)
        self.priors, self.offsets = priors, offsets
        self.off_t = [torch.tensor([a, b, 0.0], dtype=DTYPE, device=DEVICE) for a, b in offsets]
        self.blocks = list(itertools.product(corpora, weightings, max_orders, min_dfs, topks))
        self.inner = list(itertools.product(priors, range(len(offsets)), range(len(alphas))))
        self.configs = [dict(corpus=b[0], weighting=b[1], max_order=b[2], min_df=b[3],
                             topk=None if b[4] is None else f"{b[4][1]}:{b[4][0]}", fit_prior=p,
                             offset=str(offsets[o]), alpha=alphas[a], kind=kind)
                        for b in self.blocks for p, o, a in self.inner]
        self.ext_src = ext_src

    def _masks(self, tr, ev):
        if self.ext_src is None:
            return tr, ev, None, y
        s = self.ext_src
        return (torch.cat([tr, tr[s]]), torch.cat([ev, torch.zeros_like(s, dtype=torch.bool)]),
                torch.cat([tr, torch.zeros_like(s, dtype=torch.bool)]), torch.cat([y, y[s]]))

    def _jll(self, block, tr, ev):
        ck, wt, mo, md, tk = block
        tr2, ev2, dfm, yy = self._masks(tr, ev)
        cp = CORPORA[ck]
        keep = None
        if tk is not None:
            keep = select_terms(vectorize(cp, tr2, md, mo, wt, df_mask=dfm), yy, tr2, tk[0], tk[1])
        ft = vectorize(cp, tr2, md, mo, wt, keep=keep, df_mask=dfm)
        w = None if self.weight_fn is None else self.weight_fn(tr2, yy)
        jl = nb_scores(ft, nb_weights(ft, yy, tr2, self.alpha_t, self.kind, w), ev2)
        lp = log_prior(yy, tr2, w).view(1, 1, C)
        out = []
        for p in self.priors:
            base = jl + lp if p else jl
            for o in self.off_t:
                out.append(base + o.view(1, 1, C))
        return torch.cat(out, 0)                    # (len(inner), n_ev, C)

    def score(self, tr, ev):
        yev = y[ev]
        return torch.cat([cm_batch(yev, self._jll(b, tr, ev).argmax(2)) for b in self.blocks], 0)

    def predict(self, cfg, tr, ev):
        b, i = divmod(cfg, len(self.inner))
        return self._jll(self.blocks[b], tr, ev)[i]

def first_best(f1):
    return int((f1 >= f1.max() - 1e-12).nonzero()[0])

RESULTS, SELECTIONS = {}, []

def nested_cv(v, train_filter=None):
    t0 = time.perf_counter()
    pred = torch.zeros(R, N, dtype=torch.long, device=DEVICE)
    prob = torch.zeros(R, N, C, dtype=DTYPE, device=DEVICE)
    single = len(v.configs) == 1
    for r in range(R):
        for k in range(K):
            otr, ote = Ft[r] != k, Ft[r] == k
            keep = train_filter(r, k, otr) if train_filter else torch.ones_like(otr)
            if single:
                best, inner_f1 = 0, float("nan")
            else:
                cm = 0.0
                for q in range(Q):
                    iv = otr & (It[r, k] == q)
                    cm = cm + v.score(otr & ~iv & keep, iv)
                f = score_macro(cm)
                best, inner_f1 = first_best(f), float(f.max())
            jl = v.predict(best, otr & keep, ote)
            pred[r, ote] = jl.argmax(1)
            prob[r, ote] = torch.softmax(jl, 1)
            SELECTIONS.append(dict(variant=v.name, repeat=r, fold=k, cfg=best, inner_macro_f1=inner_f1, **v.configs[best]))
    return pred, prob, time.perf_counter() - t0

def register(name, pred, prob=None, runtime=float("nan"), note=""):
    per = []
    for r in range(R):
        cm = cm_batch(y, pred[r][None])
        per.append((float(score_macro(cm)), *macro_f1_cm(cm)[0].tolist(), float((pred[r] == y).double().mean())))
    per = np.array(per)
    RESULTS[name] = dict(pred=pred, prob=prob)
    row = dict(variant=name, macro_f1_mean=per[:, 0].mean(), macro_f1_std=per[:, 0].std(ddof=1),
               f1_negative=per[:, 1].mean(), f1_neutral=per[:, 2].mean(), f1_positive=per[:, 3].mean(),
               accuracy=per[:, 4].mean(), n_repeats=R, n_folds=R * K, runtime_s=runtime, note=note)
    b = boot_scores(pred).cpu().numpy()
    row["macro_f1_ci95_lo"], row["macro_f1_ci95_hi"] = np.percentile(b, [2.5, 97.5])
    out = pd.DataFrame({"Comment ID": dev["Comment ID"], "y_true": dev["Final Label"]})
    for r in range(R):
        out[f"pred_r{r}"] = [LABELS[i] for i in pred[r].tolist()]
    out.to_csv(EXP / "oof" / f"{name}.csv.gz", index=False)
    return row

TABLE, COMPARE = [], []

def run(v, note="", train_filter=None):
    pred, prob, t = nested_cv(v, train_filter)
    row = register(v.name, pred, prob, t, note)
    TABLE.append(row)
    print(f"{v.name:14s} macro-F1 {row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f} | "
          f"Neg {row['f1_negative']:.3f} Neu {row['f1_neutral']:.3f} Pos {row['f1_positive']:.3f} | acc {row['accuracy']:.3f} | "
          f"{len(v.configs)} cấu hình, {t:.1f}s")
    return row

def compare(a, b, exp=""):
    pa, pb = RESULTS[a]["pred"], RESULTS[b]["pred"]
    fa = float(np.mean([float(score_macro(cm_batch(y, pa[r][None]))) for r in range(R)]))
    fb = float(np.mean([float(score_macro(cm_batch(y, pb[r][None]))) for r in range(R)]))
    d = (boot_scores(pa) - boot_scores(pb)).cpu().numpy()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    p = max(p, 1 / len(d))
    row = dict(experiment=exp, variant=a, reference=b, f1_variant=fa, f1_reference=fb, delta=fa - fb,
               ci95_lo=lo, ci95_hi=hi, p_boot=p, accepted=bool(lo > 0))
    COMPARE.append(row)
    print(f"{exp:5s} {a} vs {b}: Δ = {fa - fb:+.4f} CI95 [{lo:+.4f}, {hi:+.4f}] p={p:.4f} -> {'CHẤP NHẬN' if lo > 0 else 'không chấp nhận'}")
    return row

def save_tables():
    pd.DataFrame(TABLE).to_csv(EXP / "results_cv.csv", index=False)
    c = pd.DataFrame(COMPARE)
    if len(c):
        o = c["p_boot"].argsort().values
        adj, cur = np.zeros(len(c)), 0.0
        for i, j in enumerate(o):
            cur = max(cur, min(1.0, (len(c) - i) * c["p_boot"].values[j]))
            adj[j] = cur
        c["p_holm"] = adj
        c.to_csv(EXP / "comparisons_cv.csv", index=False)
    pd.DataFrame(SELECTIONS).to_csv(EXP / "selected_configs.csv", index=False)

## 1. Đối chứng cũ tính lại trên cùng fold

In [7]:
R0 = NBVariant("R0", max_orders=(2,), min_dfs=(2,), alphas=(0.01,), priors=(False,))
run(R0, "cố định, không tune")

maj = torch.full((R, N), 0, dtype=torch.long, device=DEVICE)
for r in range(R):
    for k in range(K):
        otr, ote = Ft[r] != k, Ft[r] == k
        maj[r, ote] = torch.bincount(y[otr], minlength=C).argmax()
TABLE.append(register("Majority", maj, None, 0.0, "luôn đoán lớp đa số của outer-train"))
print("Majority macro-F1:", round(TABLE[-1]["macro_f1_mean"], 4))

R0             macro-F1 0.5928 ± 0.0046 | Neg 0.516 Neu 0.470 Pos 0.792 | acc 0.677 | 1 cấu hình, 0.4s
Majority macro-F1: 0.2516


In [8]:
def csr_from(ft, n_docs):
    cols = torch.full((ft.vm.numel(),), -1, device=DEVICE)
    cols[ft.vm] = torch.arange(int(ft.vm.sum()), device=DEVICE)
    return sps.csr_matrix((ft.v.cpu().numpy(), (ft.d.cpu().numpy(), cols[ft.t].cpu().numpy())),
                          shape=(n_docs, int(ft.vm.sum())))

LR_C = (0.1, 0.3, 1, 3, 10, 30)
SVC_C = (0.03, 0.1, 0.3, 1, 3, 10)

def make_model(kind, c, cw):
    if kind == "logreg":
        return LogisticRegression(C=c, class_weight=cw, max_iter=2000)
    return LinearSVC(C=c, class_weight=cw, max_iter=20000, random_state=0)

def sk_grid_task(Xtr, ytr, Xev, yev, kind, grid):
    out = []
    for c, cw in grid:
        p = make_model(kind, c, cw).fit(Xtr, ytr).predict(Xev)
        out.append(np.bincount(yev * C + p, minlength=C * C))
    return np.array(out)

def sk_final_task(Xtr, ytr, Xev, kind, c, cw):
    m = make_model(kind, c, cw).fit(Xtr, ytr)
    s = m.predict_proba(Xev) if kind == "logreg" else m.decision_function(Xev)
    return m.predict(Xev), s

def sk_nested(name, kind, Cs, corpus_key="clean", max_orders=(1, 2), min_dfs=(1, 2, 3), cws=(None, "balanced"), note=""):
    t0 = time.perf_counter()
    cp = CORPORA[corpus_key]
    blocks = list(itertools.product(max_orders, min_dfs))
    grid = list(itertools.product(Cs, cws))
    tasks, keys = [], []
    for r in range(R):
        for k in range(K):
            otr = Ft[r] != k
            for q in range(Q):
                iv = otr & (It[r, k] == q)
                it = otr & ~iv
                for bi, (mo, md) in enumerate(blocks):
                    X = csr_from(vectorize(cp, it, md, mo, "tfidf"), N)
                    a, b = it.cpu().numpy(), iv.cpu().numpy()
                    tasks.append(delayed(sk_grid_task)(X[a], y_np[a], X[b], y_np[b], kind, grid))
                    keys.append((r, k, bi))
    res = Parallel(n_jobs=N_JOBS)(tasks)
    agg = {}
    for key, cm in zip(keys, res):
        agg[key] = agg.get(key, 0) + cm
    finals, meta = [], []
    for r in range(R):
        for k in range(K):
            f = torch.cat([score_macro(torch.tensor(agg[(r, k, bi)], dtype=DTYPE, device=DEVICE).view(-1, C, C)) for bi in range(len(blocks))])
            best = first_best(f)
            bi, gi = divmod(best, len(grid))
            mo, md = blocks[bi]
            c, cw = grid[gi]
            otr = Ft[r] != k
            X = csr_from(vectorize(cp, otr, md, mo, "tfidf"), N)
            a = otr.cpu().numpy()
            finals.append(delayed(sk_final_task)(X[a], y_np[a], X[~a], kind, c, cw))
            meta.append((r, k, ~a))
            SELECTIONS.append(dict(variant=name, repeat=r, fold=k, cfg=best, inner_macro_f1=float(f.max()), corpus=corpus_key,
                                   max_order=mo, min_df=md, C=c, class_weight=str(cw), kind=kind))
    out = Parallel(n_jobs=N_JOBS)(finals)
    pred = torch.zeros(R, N, dtype=torch.long, device=DEVICE)
    prob = torch.zeros(R, N, C, dtype=DTYPE, device=DEVICE)
    for (r, k, mask), (p, s) in zip(meta, out):
        m = torch.tensor(mask, device=DEVICE)
        pred[r, m] = torch.tensor(p, device=DEVICE)
        prob[r, m] = torch.tensor(s, dtype=DTYPE, device=DEVICE)
    t = time.perf_counter() - t0
    row = register(name, pred, prob, t, note)
    TABLE.append(row)
    print(f"{name:14s} macro-F1 {row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f} | Neg {row['f1_negative']:.3f} "
          f"Neu {row['f1_neutral']:.3f} Pos {row['f1_positive']:.3f} | acc {row['accuracy']:.3f} | {t:.1f}s")
    return row

sk_nested("LogReg_C3", "logreg", (3,), max_orders=(2,), min_dfs=(2,), cws=(None,), note="sklearn, cố định C=3, không class_weight")
sk_nested("SVC_C1", "svc", (1,), max_orders=(2,), min_dfs=(2,), cws=(None,), note="sklearn, cố định C=1, không class_weight")
sk_nested("LogReg_tuned", "logreg", LR_C, note="sklearn, nested")
sk_nested("SVC_tuned", "svc", SVC_C, note="sklearn, nested")
save_tables()

LogReg_C3      macro-F1 0.5690 ± 0.0037 | Neg 0.457 Neu 0.448 Pos 0.802 | acc 0.692 | 3.2s


SVC_C1         macro-F1 0.5826 ± 0.0049 | Neg 0.491 Neu 0.453 Pos 0.804 | acc 0.690 | 1.4s


LogReg_tuned   macro-F1 0.6193 ± 0.0053 | Neg 0.544 Neu 0.525 Pos 0.790 | acc 0.682 | 41.0s


SVC_tuned      macro-F1 0.6008 ± 0.0091 | Neg 0.517 Neu 0.484 Pos 0.801 | acc 0.688 | 14.2s


In [9]:
R1 = NBVariant("R1")
assert len(R1.configs) == 84
run(R1, "nested trên lưới G")
compare("R1", "R0", "E1")
compare("LogReg_tuned", "R0", "ref")
compare("LogReg_tuned", "R1", "ref")
compare("SVC_tuned", "R1", "ref")
compare("R0", "Majority", "ref")
sel = pd.DataFrame([s for s in SELECTIONS if s["variant"] == "R1"])
print(sel[["alpha", "fit_prior", "min_df", "max_order"]].value_counts().head(10))
save_tables()

R1             macro-F1 0.6128 ± 0.0065 | Neg 0.552 Neu 0.494 Pos 0.792 | acc 0.683 | 84 cấu hình, 6.4s
E1    R1 vs R0: Δ = +0.0200 CI95 [+0.0079, +0.0327] p=0.0016 -> CHẤP NHẬN
ref   LogReg_tuned vs R0: Δ = +0.0265 CI95 [+0.0087, +0.0451] p=0.0048 -> CHẤP NHẬN
ref   LogReg_tuned vs R1: Δ = +0.0065 CI95 [-0.0071, +0.0203] p=0.3400 -> không chấp nhận
ref   SVC_tuned vs R1: Δ = -0.0120 CI95 [-0.0265, +0.0024] p=0.1064 -> không chấp nhận
ref   R0 vs Majority: Δ = +0.3412 CI95 [+0.3169, +0.3645] p=0.0002 -> CHẤP NHẬN
alpha  fit_prior  min_df  max_order
0.10   False      2       2            10
0.30   False      3       2             7
                  2       2             4
0.10   False      3       2             2
0.03   False      2       2             1
                  1       2             1
Name: count, dtype: int64


In [10]:
def w_balanced(tr, yy):
    cc = torch.bincount(yy[tr], minlength=C).to(DTYPE)
    return (tr.sum().to(DTYPE) / (C * cc))[yy]

corpus("old", lambda: [word_terms(tokens(t)) for t in dev["Clean Text Old"]])
run(NBVariant("X_old_bal", corpora=("old",), weight_fn=w_balanced), "cấu hình thăm dò của đợt trước")
compare("X_old_bal", "R1", "ref")

X_old_bal      macro-F1 0.6391 ± 0.0047 | Neg 0.579 Neu 0.547 Pos 0.792 | acc 0.695 | 84 cấu hình, 1.1s
ref   X_old_bal vs R1: Δ = +0.0263 CI95 [+0.0113, +0.0411] p=0.0004 -> CHẤP NHẬN


{'experiment': 'ref',
 'variant': 'X_old_bal',
 'reference': 'R1',
 'f1_variant': 0.6390983689233066,
 'f1_reference': 0.612802394656591,
 'delta': 0.026295974266715594,
 'ci95_lo': np.float64(0.011317224645067436),
 'ci95_hi': np.float64(0.04113438396972909),
 'p_boot': np.float64(0.0004),
 'accepted': True}

## 2. Nạp dữ liệu v2

Hash được kiểm tra theo `models/Naive_Bayes/datasets/v2/MANIFEST.json`. `test_v2.csv` không được mở trong notebook này.

In [11]:
V2 = NBDIR / "datasets" / "v2"
MAN = json.loads((V2 / "MANIFEST.json").read_text(encoding="utf-8"))
for f_ in ("film_clean_v2.csv", "pool_vsmec.csv", "pool_vsfc.csv", "label_fixes_train.csv"):
    assert hashlib.sha256((V2 / f_).read_bytes()).hexdigest() == MAN["sha256"][f_], f_

film = pd.read_csv(V2 / "film_clean_v2.csv", encoding="utf-8-sig", keep_default_na=False).set_index("Comment ID").loc[dev["Comment ID"]]
assert (film["label"].values == dev["Final Label"].values).all()
pool_me = pd.read_csv(V2 / "pool_vsmec.csv", encoding="utf-8-sig", keep_default_na=False)
pool_fc = pd.read_csv(V2 / "pool_vsfc.csv", encoding="utf-8-sig", keep_default_na=False)
TOK2 = re.compile(r"(?u)\b\w+\b")               # giữ token một ký tự để không mất chữ số
w2 = lambda t: word_terms(TOK2.findall(t))
F2 = [w2(t) for t in film["text_v2"]]
ME2, FC2 = [w2(t) for t in pool_me["text_v2"]], [w2(t) for t in pool_fc["text_v2"]]
corpus("v2", lambda: F2)
corpus("v2_me", lambda: F2 + ME2)
corpus("v2_fc", lambda: F2 + FC2)
corpus("v2_all", lambda: F2 + ME2 + FC2)
Y_ME = torch.tensor(pool_me["label"].map(L2I).values, device=DEVICE)
Y_FC = torch.tensor(pool_fc["label"].map(L2I).values, device=DEVICE)
Y_ALL = torch.cat([Y_ME, Y_FC])
print({k: CORPORA[k].n_terms for k in ("v2", "v2_me", "v2_fc", "v2_all")})
print("Film dev:", dict(zip(LABELS, np.bincount(y_np))), "| + VSMEC:", dict(zip(LABELS, np.bincount(y_np) + np.bincount(Y_ME.cpu().numpy()))))

{'v2': 29433, 'v2_me': 69104, 'v2_fc': 65004, 'v2_all': 102284}
Film dev: {'Negative': np.int64(317), 'Neutral': np.int64(392), 'Positive': np.int64(1091)} | + VSMEC: {'Negative': np.int64(3037), 'Neutral': np.int64(1412), 'Positive': np.int64(2645)}


## 3. Lớp NB có pool

Tài liệu gồm $N$ dòng phim rồi đến $M$ dòng pool. Dòng pool **luôn** ở phần train và **không bao giờ** ở phần chấm. Vocabulary và IDF dùng cả pool.

Mỗi dòng pool có trọng số $\lambda$ trong tổng theo lớp, nên đóng góp của nó vào $\theta_{ct}$ được giảm so với dòng phim. $\lambda$ được chọn ở inner CV.

In [12]:
LAMS = (0.03, 0.1, 0.3, 1.0)

class NBv2:
    def __init__(self, name, corpus_key="v2", pool_y=None, lambdas=(1.0,), balanced=False, max_orders=(1, 2), min_dfs=(1, 2, 3),
                 alphas=ALPHAS, priors=(False, True), y_train=None):
        self.name, self.ck, self.pool_y, self.balanced, self.alphas, self.priors = name, corpus_key, pool_y, balanced, alphas, priors
        self.M = 0 if pool_y is None else pool_y.numel()
        self.alpha_t = torch.tensor(alphas, dtype=DTYPE, device=DEVICE)
        self.y_fit = y if y_train is None else y_train
        self.blocks = list(itertools.product(max_orders, min_dfs, lambdas))
        self.inner = list(itertools.product(priors, range(len(alphas))))
        self.configs = [dict(corpus=corpus_key, max_order=mo, min_df=md_, lam=l, fit_prior=p, alpha=alphas[a], balanced=balanced, kind="mnb")
                        for mo, md_, l in self.blocks for p, a in self.inner]

    def _masks(self, tr, ev):
        if not self.M:
            return tr, ev, self.y_fit
        one = torch.ones(self.M, dtype=torch.bool, device=DEVICE)
        return torch.cat([tr, one]), torch.cat([ev, ~one]), torch.cat([self.y_fit, self.pool_y])

    def _weights(self, tr2, yy, lam):
        w = torch.ones(yy.numel(), dtype=DTYPE, device=DEVICE)
        if self.M:
            w[N:] = lam
        if self.balanced:
            cc = torch.zeros(C, dtype=DTYPE, device=DEVICE).index_add_(0, yy[tr2], w[tr2])
            w = w * (w[tr2].sum() / (C * cc))[yy]
        return w

    def _jll(self, block, tr, ev):
        mo, md_, lam = block
        tr2, ev2, yy = self._masks(tr, ev)
        ft = vectorize(CORPORA[self.ck], tr2, md_, mo, "tfidf")
        w = self._weights(tr2, yy, lam)
        jl = nb_scores(ft, nb_weights(ft, yy, tr2, self.alpha_t, "mnb", w), ev2)
        lp = log_prior(yy, tr2, w).view(1, 1, C)
        return torch.cat([jl + lp if p else jl for p in self.priors], 0)

    def score(self, tr, ev):
        return torch.cat([cm_batch(y[ev], self._jll(b, tr, ev).argmax(2)) for b in self.blocks], 0)

    def predict(self, cfg, tr, ev):
        b, i = divmod(cfg, len(self.inner))
        return self._jll(self.blocks[b], tr, ev)[i]

# Kiểm tra: pool không xuất hiện ở phần chấm, và lớp mới khớp NBVariant khi không có pool, cùng corpus
chk = NBv2("chk", corpus_key="clean", max_orders=(2,), min_dfs=(2,), alphas=(0.01,), priors=(False,))
a, b = Ft[0] != 0, Ft[0] == 0
assert (chk.predict(0, a, b).argmax(1) == R0.predict(0, a, b).argmax(1)).all()
tmp = NBv2("chk2", corpus_key="v2_me", pool_y=Y_ME)
t2, e2, yy2 = tmp._masks(a, b)
assert int(e2[N:].sum()) == 0 and int(t2[N:].sum()) == Y_ME.numel() and e2.numel() == CORPORA["v2_me"].n_docs
print("Lớp NBv2 khớp NBVariant khi không có pool; pool không vào phần chấm")

Lớp NBv2 khớp NBVariant khi không có pool; pool không vào phần chấm


## 4. Chuỗi biến thể NB

Theo bản đăng ký: V1 luôn làm nền vì chỉ `clean_v2` áp dụng được cho dữ liệu ngoài. Các bước sau chỉ được giữ nếu cận dưới CI95 > 0.

In [13]:
run(NBv2("V1"), "NB trên clean_v2")
compare("V1", "R1", "V1")
V_BEST, BASE_BAL = "V1", False
run(NBv2("V2", balanced=True), "V1 + class weight balanced")
if compare("V2", "V1", "V2")["accepted"]:
    V_BEST, BASE_BAL = "V2", True
print("Nền:", V_BEST)

v3 = [("V3a", "v2_me", Y_ME, "pool VSMEC"), ("V3b", "v2_fc", Y_FC, "pool VSFC"), ("V3c", "v2_all", Y_ALL, "cả hai pool")]
OBJ = {}
for nm, ck, yp, note in v3:
    OBJ[nm] = NBv2(nm, corpus_key=ck, pool_y=yp, lambdas=LAMS, balanced=BASE_BAL)
    run(OBJ[nm], note)
rows = [compare(nm, V_BEST, nm) for nm, *_ in v3]
ok = [r for r in rows if r["accepted"]]
BEST_POOL = max(ok, key=lambda r: r["delta"])["variant"] if ok else None
NB_NAME = BEST_POOL or V_BEST
NB_OBJ = OBJ[BEST_POOL] if BEST_POOL else NBv2("tmp", balanced=BASE_BAL)
print("Sau bước pool:", NB_NAME)
sel = pd.DataFrame([s for s in SELECTIONS if s["variant"] in ("V3a", "V3b", "V3c")])
print(sel.groupby("variant")["lam"].value_counts().to_dict())

fixes = pd.read_csv(V2 / "label_fixes_train.csv", encoding="utf-8-sig", keep_default_na=False)
y_fix = y.clone()
idx_of = {c: i for i, c in enumerate(dev["Comment ID"])}
for _, r in fixes.iterrows():
    y_fix[idx_of[int(r["comment_id"])]] = L2I[r["nhan_de_xuat"]]
print("Số nhãn đổi:", int((y_fix != y).sum()))
args = dict(corpus_key=NB_OBJ.ck, pool_y=NB_OBJ.pool_y, lambdas=LAMS if NB_OBJ.M else (1.0,), balanced=BASE_BAL)
run(NBv2("V4", y_train=y_fix, **args), "8 nhãn sửa, chỉ ở phần train của fold, chấm trên nhãn gốc")
if compare("V4", NB_NAME, "V4")["accepted"]:
    NB_NAME = "V4"
NB_FINAL_OBJ = NBv2(NB_NAME, y_train=y_fix if NB_NAME == "V4" else None, **args)
print("NB_v2:", NB_NAME, args["corpus_key"], "balanced =", BASE_BAL)
save_tables()

V1             macro-F1 0.6241 ± 0.0045 | Neg 0.550 Neu 0.522 Pos 0.800 | acc 0.696 | 84 cấu hình, 7.0s
V1    V1 vs R1: Δ = +0.0113 CI95 [-0.0029, +0.0256] p=0.1236 -> không chấp nhận


V2             macro-F1 0.6411 ± 0.0068 | Neg 0.579 Neu 0.550 Pos 0.794 | acc 0.697 | 84 cấu hình, 8.5s
V2    V2 vs V1: Δ = +0.0170 CI95 [+0.0073, +0.0269] p=0.0004 -> CHẤP NHẬN
Nền: V2


V3a            macro-F1 0.6449 ± 0.0117 | Neg 0.586 Neu 0.552 Pos 0.797 | acc 0.701 | 336 cấu hình, 26.3s


V3b            macro-F1 0.6345 ± 0.0093 | Neg 0.560 Neu 0.550 Pos 0.794 | acc 0.697 | 336 cấu hình, 34.2s


V3c            macro-F1 0.6453 ± 0.0033 | Neg 0.578 Neu 0.558 Pos 0.800 | acc 0.704 | 336 cấu hình, 29.9s
V3a   V3a vs V2: Δ = +0.0038 CI95 [-0.0043, +0.0118] p=0.3584 -> không chấp nhận
V3b   V3b vs V2: Δ = -0.0066 CI95 [-0.0172, +0.0043] p=0.2532 -> không chấp nhận
V3c   V3c vs V2: Δ = +0.0042 CI95 [-0.0071, +0.0153] p=0.4624 -> không chấp nhận
Sau bước pool: V2
{('V3a', 0.03): 25, ('V3b', 0.03): 25, ('V3c', 0.03): 25}
Số nhãn đổi: 8


V4             macro-F1 0.6386 ± 0.0101 | Neg 0.574 Neu 0.553 Pos 0.789 | acc 0.693 | 84 cấu hình, 8.5s
V4    V4 vs V2: Δ = -0.0025 CI95 [-0.0078, +0.0027] p=0.3340 -> không chấp nhận
NB_v2: V2 v2 balanced = True


## 5. LogReg trên clean_v2, có và không có pool (sklearn, CPU)

In [14]:
LR2_C, LR2_CW, LR2_MD = (0.3, 1, 3, 10), (None, "balanced"), (1, 2)

def lr2_grid(Xtr, ytr, film_n, Xev, yev, lams):
    out = []
    for lam in lams:
        w = np.ones(len(ytr)); w[film_n:] = lam
        for c, cw in itertools.product(LR2_C, LR2_CW):
            p = LogisticRegression(C=c, class_weight=cw, max_iter=2000).fit(Xtr, ytr, sample_weight=w).predict(Xev)
            out.append(np.bincount(yev * C + p, minlength=C * C))
    return np.array(out)

def lr2_final(Xtr, ytr, film_n, Xev, lam, c, cw):
    w = np.ones(len(ytr)); w[film_n:] = lam
    return LogisticRegression(C=c, class_weight=cw, max_iter=2000).fit(Xtr, ytr, sample_weight=w).predict_proba(Xev)

def x_parts(ck, tr, ev, md_):
    cp = CORPORA[ck]
    M = cp.n_docs - N
    t2 = torch.cat([tr, torch.ones(M, dtype=torch.bool, device=DEVICE)])
    X = csr_from(vectorize(cp, t2, md_, 2, "tfidf"), cp.n_docs).tocsr()
    a, b = np.where(tr.cpu().numpy())[0], np.where(ev.cpu().numpy())[0]
    return sps.vstack([X[a], X[N:]]).tocsr(), X[b], len(a)

def lr2_nested(name, ck, ypool, lams, note):
    t0 = time.perf_counter()
    yp = np.array([], dtype=np.int64) if ypool is None else ypool.cpu().numpy()
    grid = [(l, c, cw) for l in lams for c, cw in itertools.product(LR2_C, LR2_CW)]
    tasks, keys = [], []
    for r in range(R):
        for k in range(K):
            otr = Ft[r] != k
            for q in range(Q):
                iv = otr & (It[r, k] == q)
                it = otr & ~iv
                for mi, md_ in enumerate(LR2_MD):
                    Xa, Xb, nf = x_parts(ck, it, iv, md_)
                    ya = np.r_[y_np[it.cpu().numpy()], yp]
                    tasks.append(delayed(lr2_grid)(Xa, ya, nf, Xb, y_np[iv.cpu().numpy()], lams)); keys.append((r, k, mi))
    agg = {}
    for key, cm in zip(keys, Parallel(n_jobs=N_JOBS)(tasks)):
        agg[key] = agg.get(key, 0) + cm
    finals, meta = [], []
    for r in range(R):
        for k in range(K):
            f = torch.cat([score_macro(torch.tensor(agg[(r, k, mi)], dtype=DTYPE, device=DEVICE).view(-1, C, C)) for mi in range(len(LR2_MD))])
            best = first_best(f)
            mi, gi = divmod(best, len(grid))
            l, c, cw = grid[gi]
            otr, ote = Ft[r] != k, Ft[r] == k
            Xa, Xb, nf = x_parts(ck, otr, ote, LR2_MD[mi])
            finals.append(delayed(lr2_final)(Xa, np.r_[y_np[otr.cpu().numpy()], yp], nf, Xb, l, c, cw))
            meta.append((r, ote))
            SELECTIONS.append(dict(variant=name, repeat=r, fold=k, cfg=best, inner_macro_f1=float(f.max()), corpus=ck, max_order=2,
                                   min_df=LR2_MD[mi], lam=l, C=c, class_weight=str(cw), kind="logreg"))
    pred = torch.zeros(R, N, dtype=torch.long, device=DEVICE)
    prob = torch.zeros(R, N, C, dtype=DTYPE, device=DEVICE)
    for (r, m), s in zip(meta, Parallel(n_jobs=N_JOBS)(finals)):
        st = torch.tensor(s, dtype=DTYPE, device=DEVICE)
        pred[r, m], prob[r, m] = st.argmax(1), st
    t = time.perf_counter() - t0
    row = register(name, pred, prob, t, note)
    TABLE.append(row)
    print(f"{name:14s} macro-F1 {row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f} | Neg {row['f1_negative']:.3f} Neu {row['f1_neutral']:.3f} "
          f"Pos {row['f1_positive']:.3f} | acc {row['accuracy']:.3f} | {t:.1f}s")

lr2_nested("LR1", "v2", None, (1.0,), "LogReg trên clean_v2; sklearn, CPU")
lr2_nested("LR2", "v2_me", Y_ME, (0.1, 0.3, 1.0), "LR1 + pool VSMEC; sklearn, CPU")
compare("LR1", "LogReg_tuned", "LR1")
LR_NAME = "LR2" if compare("LR2", "LR1", "LR2")["accepted"] else "LR1"
LR_CK, LR_POOL = ("v2_me", Y_ME) if LR_NAME == "LR2" else ("v2", None)
print("LogReg_v2:", LR_NAME)
save_tables()

LR1            macro-F1 0.6416 ± 0.0115 | Neg 0.566 Neu 0.557 Pos 0.801 | acc 0.702 | 23.5s


LR2            macro-F1 0.6159 ± 0.0044 | Neg 0.502 Neu 0.535 Pos 0.810 | acc 0.705 | 589.7s
LR1   LR1 vs LogReg_tuned: Δ = +0.0222 CI95 [+0.0083, +0.0362] p=0.0020 -> CHẤP NHẬN
LR2   LR2 vs LR1: Δ = -0.0257 CI95 [-0.0436, -0.0082] p=0.0028 -> không chấp nhận
LogReg_v2: LR1


## 6. HV2: soft voting NB + LogReg

In [15]:
from sklearn.metrics import f1_score

def sel_of(name):
    return {(s["repeat"], s["fold"]): s for s in SELECTIONS if s["variant"] == name}

def hv2(nb_name, nb_var, lr_name, ck, ypool):
    t0 = time.perf_counter()
    s_nb, s_lr = sel_of(nb_name), sel_of(lr_name)
    yp = np.array([], dtype=np.int64) if ypool is None else ypool.cpu().numpy()
    P_nb, tasks, keys = {}, [], []
    for r in range(R):
        for k in range(K):
            otr = Ft[r] != k
            c = s_lr[(r, k)]
            cw = None if c["class_weight"] == "None" else c["class_weight"]
            for q in range(Q):
                iv = otr & (It[r, k] == q)
                it = otr & ~iv
                P_nb[(r, k, q)] = torch.softmax(nb_var.predict(int(s_nb[(r, k)]["cfg"]), it, iv), 1).cpu().numpy()
                Xa, Xb, nf = x_parts(ck, it, iv, int(c["min_df"]))
                tasks.append(delayed(lr2_final)(Xa, np.r_[y_np[it.cpu().numpy()], yp], nf, Xb, c["lam"], c["C"], cw)); keys.append((r, k, q))
    P_lr = dict(zip(keys, Parallel(n_jobs=N_JOBS)(tasks)))
    WS = (0.0, 0.25, 0.5, 0.75, 1.0)
    pa_all, pb_all = RESULTS[nb_name]["prob"].cpu().numpy(), RESULTS[lr_name]["prob"].cpu().numpy()
    pv, pva = np.zeros((R, N), dtype=np.int64), np.zeros((R, N, C))
    for r in range(R):
        for k in range(K):
            A = np.vstack([P_nb[(r, k, q)] for q in range(Q)])
            B = np.vstack([P_lr[(r, k, q)] for q in range(Q)])
            yy = np.concatenate([y_np[(F_np[r] != k) & (I_np[r, k] == q)] for q in range(Q)])
            fs = [f1_score(yy, (w * A + (1 - w) * B).argmax(1), average="macro") for w in WS]
            w = WS[int(np.argmax(fs))]
            ote = F_np[r] == k
            pva[r, ote] = w * pa_all[r, ote] + (1 - w) * pb_all[r, ote]
            pv[r, ote] = pva[r, ote].argmax(1)
            SELECTIONS.append(dict(variant="HV2", repeat=r, fold=k, cfg=-1, inner_macro_f1=max(fs), w_nb=w, kind="softvote"))
    t = time.perf_counter() - t0
    row = register("HV2", torch.tensor(pv, device=DEVICE), torch.tensor(pva, device=DEVICE), t, "soft voting NB_v2 + LogReg_v2")
    TABLE.append(row)
    print(f"HV2            macro-F1 {row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f} | Neg {row['f1_negative']:.3f} Neu {row['f1_neutral']:.3f} "
          f"Pos {row['f1_positive']:.3f} | acc {row['accuracy']:.3f} | {t:.1f}s")

hv2(NB_NAME, NB_FINAL_OBJ, LR_NAME, LR_CK, LR_POOL)
compare("HV2", NB_NAME, "HV2")
compare("HV2", LR_NAME, "ref")
compare("HV2", "R1", "ref")
compare("HV2", "X_old_bal", "ref")
print(pd.DataFrame([s for s in SELECTIONS if s["variant"] == "HV2"])["w_nb"].value_counts().sort_index().to_dict())
save_tables()

HV2            macro-F1 0.6481 ± 0.0085 | Neg 0.586 Neu 0.556 Pos 0.803 | acc 0.706 | 1.7s
HV2   HV2 vs V2: Δ = +0.0070 CI95 [-0.0009, +0.0147] p=0.0792 -> không chấp nhận
ref   HV2 vs LR1: Δ = +0.0065 CI95 [-0.0025, +0.0157] p=0.1724 -> không chấp nhận
ref   HV2 vs R1: Δ = +0.0353 CI95 [+0.0196, +0.0509] p=0.0002 -> CHẤP NHẬN
ref   HV2 vs X_old_bal: Δ = +0.0090 CI95 [-0.0061, +0.0247] p=0.2448 -> không chấp nhận
{0.0: 2, 0.25: 6, 0.5: 10, 0.75: 7}
